# CLIP — Learning Transferable Visual Models From Natural Language Supervision (2021)

_Papers · Self-supervised & Representation_

**Train an image encoder and a text encoder to agree, then classify any image by matching it to text prompts — zero labels at test time.**

---

This notebook is a practice scaffold for the **CLIP — Learning Transferable Visual Models From Natural Language Supervision (2021)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np
torch.manual_seed(0); np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- 0. Sanity-check the worked example: symmetric CLIP loss on a 2x2 similarity matrix. ---
# S rows = images, cols = texts; diagonal = true pairs; temperature tau = 0.5  ->  logits = S/tau.
S = torch.tensor([[0.9, 0.1],
                  [0.2, 0.8]])
tau = 0.5
logits = S / tau                                    # N x N scaled cosine-similarity (image rows)
labels = torch.arange(S.shape[0])                   # diagonal is the positive: [0, 1, ..., N-1]
loss_i2t = F.cross_entropy(logits,     labels)      # image -> text  (across rows)
loss_t2i = F.cross_entropy(logits.t(), labels)      # text  -> image (across columns)
clip_loss = (loss_i2t + loss_t2i) / 2
print("worked example:  i2t =", round(loss_i2t.item(),4),
      " t2i =", round(loss_t2i.item(),4),
      " symmetric =", round(clip_loss.item(),4))
# worked example:  i2t = 0.2236  t2i = 0.2204  symmetric = 0.222


# --- 1. Toy cross-modal data: C classes, each with a hidden prototype. ---
# An "image" = class prototype (in image space) + noise; its "text" = class-name vector (text space).
# The two spaces have DIFFERENT dimensions, so the encoders must learn a shared alignment.
C, IMG_DIM, TXT_DIM, D = 8, 32, 24, 16              # classes, image-dim, text-dim, shared embed dim
img_proto = torch.randn(C, IMG_DIM)                 # hidden image prototype per class
txt_proto = torch.randn(C, TXT_DIM)                 # the class-name (text) vector per class

def sample(n_per_class, noise=0.6):
    ys = torch.arange(C).repeat_interleave(n_per_class)
    imgs = img_proto[ys] + noise * torch.randn(len(ys), IMG_DIM)   # noisy image features
    txts = txt_proto[ys]                                           # class-name text features
    return imgs, txts, ys

train_imgs, train_txts, train_y = sample(40)        # training pairs
train_imgs, train_txts = train_imgs.to(device), train_txts.to(device)


# --- 2. Two encoder towers -> shared D-dim space (built by hand from nn primitives). ---
class Encoder(nn.Module):                            # small MLP into the joint embedding space
    def __init__(self, din, d=D, hid=64):
        super().__init__(); self.l1 = nn.Linear(din, hid); self.l2 = nn.Linear(hid, d)
    def forward(self, x): return self.l2(F.relu(self.l1(x)))      # raw embedding (normalized later)

img_enc, txt_enc = Encoder(IMG_DIM).to(device), Encoder(TXT_DIM).to(device)
# learned temperature, log-parameterized as in the paper: store log(1/tau), init 1/tau = 1/0.07.
log_inv_tau = nn.Parameter(torch.tensor(np.log(1/0.07), dtype=torch.float32, device=device))
params = list(img_enc.parameters()) + list(txt_enc.parameters()) + [log_inv_tau]
opt = torch.optim.Adam(params, lr=1e-3)


# --- 3. The symmetric CLIP loss (Figure 3), built by hand. ---
def clip_step(imgs, txts):
    Ie = F.normalize(img_enc(imgs), dim=1)           # L2-normalize -> cosine = dot product
    Te = F.normalize(txt_enc(txts), dim=1)
    scale = log_inv_tau.exp().clamp(max=100.0)       # 1/tau, clipped for stability (paper)
    logits = scale * Ie @ Te.t()                     # N x N scaled cosine-similarity grid
    tgt = torch.arange(len(imgs), device=imgs.device)
    return (F.cross_entropy(logits, tgt) + F.cross_entropy(logits.t(), tgt)) / 2   # symmetric

# --- 4. Train both encoders jointly with in-batch negatives. ---
img_enc.train(); txt_enc.train(); B = 64; M = len(train_imgs)
for ep in range(300):
    perm = torch.randperm(M, device=device); tot = 0.0; nb = 0
    for s in range(0, M, B):
        bi = perm[s:s+B]
        loss = clip_step(train_imgs[bi], train_txts[bi])
        opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item(); nb += 1
    if ep % 60 == 0: print(f"  ep {ep:3d}  symmetric loss {tot/nb:.3f}  (1/tau={log_inv_tau.exp().item():.1f})")

# --- 5. ZERO-SHOT: freeze encoders; classify HELD-OUT images by matching to class-name embeddings. ---
img_enc.eval(); txt_enc.eval()
test_imgs, _, test_y = sample(50)                    # fresh held-out images, never seen in training
test_imgs = test_imgs.to(device)
with torch.no_grad():
    class_emb = F.normalize(txt_enc(txt_proto.to(device)), dim=1)   # one text embedding per class
    img_emb   = F.normalize(img_enc(test_imgs), dim=1)
    sims = img_emb @ class_emb.t()                   # cosine sim of each image to every class name
    pred = sims.argmax(1).cpu()                      # zero-shot prediction = most similar class name
acc = (pred == test_y).float().mean().item()
print(f"\nZERO-SHOT accuracy on held-out images: {acc:.3f}   (chance = {1/C:.3f})")
# Trained CLIP-style encoders match held-out images to their class name far above chance.
# (Exact numbers vary by hardware/seed; this is our small toy run, not the paper's reported result.)

## Visualize the data & results

_Does symmetric contrastive training let frozen encoders classify held-out images zero-shot, far above a random-encoder baseline?_

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, numpy as np
torch.manual_seed(0); np.random.seed(0)

# Train two encoders with the SYMMETRIC contrastive loss on toy (image, class-name) pairs, freeze
# them, then ZERO-SHOT classify held-out images by argmax cosine similarity to class-name embeddings.
# Compare trained encoders vs random (untrained) encoders vs the 1/C chance rate (toy reproduction).
C, IMG_DIM, TXT_DIM, D = 8, 32, 24, 16
img_proto = torch.randn(C, IMG_DIM); txt_proto = torch.randn(C, TXT_DIM)

def sample(n, noise=0.6):
    ys = torch.arange(C).repeat_interleave(n)
    return img_proto[ys] + noise*torch.randn(len(ys), IMG_DIM), txt_proto[ys], ys

class Encoder(nn.Module):
    def __init__(s, din, d=D, hid=64):
        super().__init__(); s.l1=nn.Linear(din,hid); s.l2=nn.Linear(hid,d)
    def forward(s, x): return s.l2(F.relu(s.l1(x)))

def clip_loss(ie, te, scale):
    Ie=F.normalize(ie,dim=1); Te=F.normalize(te,dim=1)
    logits = scale * Ie @ Te.t(); tgt = torch.arange(len(ie))
    return (F.cross_entropy(logits,tgt) + F.cross_entropy(logits.t(),tgt))/2

def zero_shot_acc(img_enc, txt_enc):
    img_enc.eval(); txt_enc.eval()
    ti, _, ty = sample(50)
    with torch.no_grad():
        ce = F.normalize(txt_enc(txt_proto), dim=1)
        ie = F.normalize(img_enc(ti), dim=1)
        pred = (ie @ ce.t()).argmax(1)
    return (pred == ty).float().mean().item()

tr_i, tr_t, _ = sample(40)
# --- trained encoders ---
ie_t, te_t = Encoder(IMG_DIM), Encoder(TXT_DIM)
log_inv_tau = nn.Parameter(torch.tensor(np.log(1/0.07)))
opt = torch.optim.Adam(list(ie_t.parameters())+list(te_t.parameters())+[log_inv_tau], lr=1e-3)
ie_t.train(); te_t.train(); B=64; M=len(tr_i)
for ep in range(300):
    perm = torch.randperm(M)
    for s in range(0, M, B):
        bi = perm[s:s+B]; scale = log_inv_tau.exp().clamp(max=100.0)
        loss = clip_loss(ie_t(tr_i[bi]), te_t(tr_t[bi]), scale)
        opt.zero_grad(); loss.backward(); opt.step()
acc_trained = zero_shot_acc(ie_t, te_t)
# --- random (untrained) encoders, same architecture ---
acc_random = zero_shot_acc(Encoder(IMG_DIM), Encoder(TXT_DIM))

print("trained", round(acc_trained,3), "random", round(acc_random,3), "chance", round(1/C,3))
# trained zero-shot accuracy >> random ~ chance: the contrastive training built the shared space.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The headline. You trained image and text encoders with the symmetric contrastive loss on toy
            (image, class-name) pairs, then froze them and classified held-out images by matching each to
            the class-name embeddings — never training a classifier head. Zero-shot accuracy is well above chance.
            What does that demonstrate, and what is the one-line ablation that would break it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Confirm zero-shot accuracy on held-out images sits well above the $1/C$ chance rate. — _Above-chance matching with no trained head means the joint image–text space generalizes to unseen images._
- Ablate: replace the trained encoders with random (untrained) ones and redo the argmax matching. — _If the alignment came from the architecture or the argmax rule rather than training, random encoders would match too. They do not — isolating the contrastive training._
- Conclude the symmetric InfoNCE training, not the architecture or the matching rule, built the shared space. — _Same encoders, same argmax, same data; only whether they were trained differs._

**Answer:** It demonstrates that the symmetric contrastive training aligned images and class-name text
                 into one shared space that generalizes: a held-out image lands nearest its own class name, so
                 plain $\arg\max$ cosine similarity classifies it — zero-shot, no head. Swapping in
                 random untrained encoders is the ablation that breaks it: matching collapses to chance,
                 isolating the InfoNCE training (not the architecture or the argmax) as the source of the
                 alignment. Our CODEVIZ panel shows trained-encoder zero-shot accuracy far above the random-encoder
                 baseline.

</details>

**Problem 2.** Your worked example gave a symmetric loss of $0.222$ with the diagonal dominating. Now suppose the
            model gets confused on pair 1: $\mathrm{sim}(I_1,T_2)$ jumps from $0.1$ to $0.95$ while
            $\mathrm{sim}(I_1,T_1)$ stays $0.9$ (keep the rest of $S$ and $\tau=0.5$). Does the image→text loss
            for row 1 go up or down, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recompute row-1 logits at $1/\tau=2$: $L_{11}=1.8$ (unchanged), $L_{12}=2\times0.95=1.9$ (was $0.2$). — _Only the off-diagonal entry in row 1 changed; scale by $1/\tau$._
- Softmax row 1: $e^{1.8}=6.0496,\ e^{1.9}=6.6859$; sum $=12.7355$. $p_{11}=6.0496/12.7355=0.4750$. — _A negative now more similar than the true text inflates the denominator and steals probability._
- Row-1 loss $=-\log(0.4750)=0.7444$, up from $0.1839$. — _Smaller probability for the correct text means larger $-\log p$._

**Answer:** The row-1 loss goes up sharply, from $0.184$ to about $0.744$ (and the overall symmetric
                 loss rises with it). A mismatched text ($T_2$) that is more similar to image 1 than its true
                 caption ($T_1$) inflates the softmax denominator, so the probability of the correct pair drops
                 ($0.83 \to 0.48$) and $-\log p$ climbs. That is exactly the pressure the contrastive loss applies:
                 push the off-diagonal (wrong) pairs below the diagonal (right) ones.

</details>

**Problem 3.** You implement the loss as cross-entropy on the rows only (image→text) and skip the column direction.
            Training "works" — the loss falls — but zero-shot matching of texts to images is poor and the
            text embeddings look degenerate. What did the missing term provide, and why does omitting it hurt?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall the loss is symmetric: $\tfrac12(\text{CE}_{\text{row}} + \text{CE}_{\text{col}})$. — _The row term trains "given an image, find its text"; the column term trains "given a text, find its image."_
- Note that with only the row term, the gradient signal to the text encoder is weaker and one-sided. — _Each text appears as a target in exactly one row, but the column term would make every text actively compete to pick its own image — a stronger, balanced signal._
- Fix: add cross-entropy on logits.t() with the same targets and average the two. — _That restores the text→image direction, so both encoders get a symmetric pull-together / push-apart signal._

**Answer:** You dropped the text→image (column) cross-entropy, so the loss became asymmetric. The
                 missing term trains the text encoder to pick the right image out of the batch; without it
                 the text side gets a weak, one-sided signal and its embeddings drift, so text→image retrieval
                 fails. Fix: compute cross-entropy on both the logits and their transpose with targets
                 $[0,\dots,N\!-\!1]$ and average — the symmetric objective of Figure 3.

</details>